# Train All Experiments (5 Models x 3 Seeds)

Models: A (single-task species), B (single-task freshness), C-EW, C-UW, C-DWA (multi-task, one loss-weighting strategy each). Checkpoints and per-run history save to Drive, not the ephemeral Colab clone, and already-checkpointed runs are skipped so an interrupted session can resume by re-running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'
CURVES_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results/figures/curves'

import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CURVES_DIR, exist_ok=True)

!unzip -q "$DATASET_ZIP" -d /content/data

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

import pandas as pd
from train import train_single_task, train_multitask, TrainConfig
from plotting import plot_single_task_curves, plot_multitask_curves

train_df = pd.read_csv('/content/repo/02_Manifests/train.csv')
val_df = pd.read_csv('/content/repo/02_Manifests/val.csv')
len(train_df), len(val_df)

In [ ]:
SEEDS = [42, 43, 44]

single_task_experiments = [
    {'name': 'ModelA_species', 'task': 'species'},
    {'name': 'ModelB_freshness', 'task': 'freshness'},
]
multitask_experiments = [
    {'name': 'ModelC_EW', 'strategy': 'EW'},
    {'name': 'ModelC_UW', 'strategy': 'UW'},
    {'name': 'ModelC_DWA', 'strategy': 'DWA'},
]

In [ ]:
import json
import matplotlib.pyplot as plt
import torch

cfg = TrainConfig()
summary_rows = []

for exp in single_task_experiments:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            print(f'[{run_name}] checkpoint exists, skipping')
            continue
        print(f'[{run_name}] starting...')
        result = train_single_task(
            exp['task'], train_df, val_df, DATASET_ROOT, ckpt_path, seed=seed, cfg=cfg, run_name=run_name
        )
        summary_rows.append({
            'run_name': run_name, 'model': exp['name'], 'seed': seed,
            'best_val_f1': result['best_val_f1'],
            'training_time_sec': result['training_time_sec'],
            'epochs_trained': len(result['history']),
            'source': 'complete',
        })
        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)
        fig = plot_single_task_curves(
            result['history'], run_name, save_path=os.path.join(CURVES_DIR, f'{run_name}.png')
        )
        plt.close(fig)

for exp in multitask_experiments:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            print(f'[{run_name}] checkpoint exists, skipping')
            continue
        print(f'[{run_name}] starting...')
        result = train_multitask(
            exp['strategy'], train_df, val_df, DATASET_ROOT, ckpt_path, seed=seed, cfg=cfg, run_name=run_name
        )
        summary_rows.append({
            'run_name': run_name, 'model': exp['name'], 'seed': seed,
            'best_val_f1': result['best_val_f1'],
            'training_time_sec': result['training_time_sec'],
            'epochs_trained': len(result['history']),
            'source': 'complete',
        })
        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)
        fig = plot_multitask_curves(
            result['history'], run_name, save_path=os.path.join(CURVES_DIR, f'{run_name}.png')
        )
        plt.close(fig)

# Backfill any run completed in a PREVIOUS session/execution of this cell -- summary_rows
# above only captures runs trained during *this* execution, but a run whose checkpoint
# already existed (skipped above) still finished at some point. Prefer the full history
# JSON when it exists; if the run was interrupted before it could return (and so never
# wrote a history file), fall back to the checkpoint's own saved epoch/F1 -- an
# incomplete reconstruction (no train/val curve, no training_time_sec), flagged as such.
all_run_names = (
    [f"{exp['name']}_seed{seed}" for exp in single_task_experiments for seed in SEEDS]
    + [f"{exp['name']}_seed{seed}" for exp in multitask_experiments for seed in SEEDS]
)
already_in_summary = {row['run_name'] for row in summary_rows}

for run_name in all_run_names:
    if run_name in already_in_summary:
        continue
    history_path = os.path.join(RESULTS_DIR, f'{run_name}_history.json')
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')

    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        f1_key = 'val_f1_mean' if 'val_f1_mean' in history[0] else 'val_f1_macro'
        best_val_f1 = max(h[f1_key] for h in history)
        epochs_trained = len(history)
        source = 'complete'
    elif os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        best_val_f1 = ckpt.get('val_f1_macro', ckpt.get('val_f1_mean'))
        epochs_trained = ckpt['epoch'] + 1  # best epoch reached; run may have gone further before disconnect
        source = 'checkpoint_only (interrupted before history was saved)'
    else:
        continue  # not trained at all yet

    model_name, seed_str = run_name.rsplit('_seed', 1)
    summary_rows.append({
        'run_name': run_name, 'model': model_name, 'seed': int(seed_str),
        'best_val_f1': best_val_f1,
        'training_time_sec': None,  # not recorded outside the session that trained it
        'epochs_trained': epochs_trained,
        'source': source,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('run_name').reset_index(drop=True)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'training_summary.csv'), index=False)
summary_df

`training_summary.csv` and each run's `*_history.json` are already saved under `RESULTS_DIR` on Drive -- that is the persistent copy. `/content/repo` is deleted when this runtime recycles, so nothing further needs to be copied there. To add these results to the GitHub repository, download them from Drive and commit them from a machine with push access.